## 🎯 Learning Objectives
* Design and implement an end-to-end image generation pipeline using modern Stable Diffusion models.
* Integrate ControlNet for precise structural or pose control within the generation process.
* Apply LoRA (Low-Rank Adaptation) for consistent style or character injection.
* Incorporate post-processing steps to enhance the quality and usability of generated images.
* Develop a modular and efficient pipeline suitable for production environments.


## Exercise: Build an End-to-End Image Generation Pipeline

**Lesson ID:** SD02-L08

In this exercise, you will design and implement a robust, end-to-end image generation pipeline. This pipeline should be capable of taking a text prompt and a control image, and producing a high-quality, post-processed image, leveraging the latest advancements in Stable Diffusion technology.

### Task Description

Your goal is to create a Python function, `generate_image_pipeline`, that encapsulates the entire image generation process. This function will simulate a production-ready workflow, integrating multiple components to achieve precise control and high-fidelity outputs.

### Requirements

1.  **Base Model Integration**: Utilize a modern Stable Diffusion model (e.g., `stabilityai/stable-diffusion-3-medium` or a similar 2026-era foundation model). Assume it's loaded efficiently.
2.  **ControlNet Integration**: Incorporate a ControlNet model (e.g., `lllyasviel/sd-controlnet-canny`) to guide the image generation based on an input control image. You'll need to preprocess the control image (e.g., Canny edge detection).
3.  **LoRA Application**: Apply a LoRA model (e.g., a custom style LoRA or character LoRA) to influence the aesthetic or content of the generated image. Assume the LoRA is loaded and applied correctly.
4.  **Text Prompting**: The pipeline must accept a positive and an optional negative text prompt.
5.  **Post-processing**: Implement at least one post-processing step. This could be: 
    *   Upscaling (e.g., using a dedicated upscaler model or a simple `PIL` resize for demonstration).
    *   Face restoration (e.g., using a mock `GFPGAN` or `CodeFormer` integration).
    *   Color correction.
    *   Adding a watermark (for simplicity, a `PIL` text overlay).
6.  **Function Signature**: The pipeline should be callable with a clear signature, e.g., `generate_image_pipeline(prompt: str, control_image: PIL.Image.Image, negative_prompt: str = "", seed: int = None, steps: int = 25) -> PIL.Image.Image`.
7.  **Efficiency Considerations**: While full optimization is beyond this exercise, consider aspects like device placement (`cuda` if available), `torch.float16` for inference, and batching (though we'll generate one image for simplicity).
8.  **Modularity**: Ensure your code is well-structured and easy to understand.

### Evaluation Criteria

*   **Correctness**: Does the pipeline correctly integrate all specified components (base model, ControlNet, LoRA, post-processing)?
*   **Functionality**: Does the `generate_image_pipeline` function run without errors and produce an output image?
*   **Clarity & Readability**: Is the code well-commented and easy to follow?
*   **Adherence to Requirements**: Are all the requirements listed above met?
*   **Production Readiness Mindset**: Does the code reflect an understanding of how such a pipeline would be structured for real-world use (e.g., clear function signature, parameterization)?


In [ ]:
import torch
from PIL import Image
import numpy as np
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from controlnet_aux import CannyDetector
import os

# --- Configuration and Mock Setup (2026 Ready) ---

# Set device to CUDA if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Mock model paths (in a real scenario, these would be actual Hugging Face model IDs or local paths)
BASE_MODEL_ID = "stabilityai/stable-diffusion-3-medium" # A powerful 2026-era foundation model
CONTROLNET_MODEL_ID = "lllyasviel/controlnet-sd3-canny" # Hypothetical SD3-compatible ControlNet
LORA_MODEL_ID = "agenticlabs/sd3-style-vaporwave-lora" # Hypothetical custom LoRA

# --- Helper Functions for Mocking and Preprocessing ---

def mock_load_model(model_id, model_type):
    """Mocks loading a model. In a real scenario, this would use from_pretrained."""
    print(f"[MOCK] Loading {model_type} from {model_id}...")
    # Simulate a dummy object that can be passed around
    class MockModel:
        def __init__(self, name):
            self.name = name
            self.device = device
            self.dtype = torch.float16 if device == "cuda" else torch.float32
        def to(self, device, dtype=None):
            self.device = device
            if dtype: self.dtype = dtype
            return self
        def __call__(self, *args, **kwargs):
            print(f"[MOCK] {self.name} called.")
            return None # Or a dummy tensor if needed

    if model_type == "pipeline":
        # For the pipeline, we need a more complex mock that can simulate generation
        class MockPipeline(MockModel):
            def __init__(self, name, controlnet, lora):
                super().__init__(name)
                self.controlnet = controlnet
                self.lora = lora
                self.scheduler = UniPCMultistepScheduler()
                self.vae = MockModel("MockVAE")
                self.text_encoder = MockModel("MockTextEncoder")
                self.unet = MockModel("MockUNet")

            def load_lora_weights(self, lora_id):
                print(f"[MOCK] Loading LoRA weights {lora_id} into pipeline.")

            def __call__(self, prompt, image, negative_prompt, num_inference_steps, generator, **kwargs):
                print(f"[MOCK] Generating image with prompt: '{prompt}' and control image.")
                # Simulate image generation: create a black image
                dummy_image = Image.new('RGB', (768, 768), color = 'black')
                return [dummy_image]
        return MockPipeline(model_id, mock_load_model(CONTROLNET_MODEL_ID, "controlnet"), mock_load_model(LORA_MODEL_ID, "lora"))
    elif model_type == "controlnet":
        return ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16) if device == "cuda" else ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny")
    elif model_type == "lora":
        return MockModel(model_id) # LoRA is typically loaded into the pipeline directly
    else:
        return MockModel(model_id)


def preprocess_control_image(image: Image.Image) -> Image.Image:
    """Applies Canny edge detection to the control image."""
    print("[PREPROCESS] Applying Canny edge detection...")
    canny_detector = CannyDetector()
    processed_image = canny_detector(image)
    return processed_image

def mock_upscale_image(image: Image.Image) -> Image.Image:
    """Mocks an image upscaling process."""
    print("[POSTPROCESS] Mock Upscaling image...")
    # In a real scenario, this would use a dedicated upscaler model (e.g., ESRGAN, SwinIR)
    # For this exercise, we'll just resize it to double the dimensions.
    upscaled_image = image.resize((image.width * 2, image.height * 2), Image.LANCZOS)
    return upscaled_image

def mock_add_watermark(image: Image.Image, text: str = "AgenticLabs.ng") -> Image.Image:
    """Mocks adding a simple watermark to the image."""
    print("[POSTPROCESS] Adding watermark...")
    from PIL import ImageDraw, ImageFont
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    try:
        font = ImageFont.truetype("arial.ttf", 30)
    except IOError:
        font = ImageFont.load_default()
    text_width, text_height = draw.textsize(text, font)
    x = img_copy.width - text_width - 10
    y = img_copy.height - text_height - 10
    draw.text((x, y), text, font=font, fill=(255, 255, 255, 128)) # White, semi-transparent
    return img_copy

# --- Load Mock ControlNet and Pipeline (for student's use) ---

# Load ControlNet model
# In a real scenario:
# controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
# For this exercise, we use the actual Canny ControlNet for preprocessing, but the pipeline will be mocked.
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
controlnet.to(device)

# Load the base pipeline with ControlNet
# In a real scenario:
# pipe = StableDiffusionControlNetPipeline.from_pretrained(
#     BASE_MODEL_ID,
#     controlnet=controlnet,
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#     safety_checker=None # Disable for faster inference if not needed in production
# )
# For this exercise, we use a mock pipeline to simulate generation without downloading large models.
pipe = mock_load_model(BASE_MODEL_ID, "pipeline")
pipe.controlnet = controlnet # Assign the actual controlnet to the mock pipe
pipe.to(device)

# Load LoRA weights into the pipeline
# In a real scenario:
# pipe.load_lora_weights(LORA_MODEL_ID)
pipe.load_lora_weights(LORA_MODEL_ID) # Mock call

# Set scheduler for faster inference
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

print("Setup complete. `pipe` and `controlnet` are ready for use.")

# --- Example Control Image (for student's use) ---
# Download a sample image for ControlNet input
control_image_url = "https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd_controlnet/person.png"
input_control_image = load_image(control_image_url).resize((768, 768))
print(f"Loaded example control image of size: {input_control_image.size}")


### Your Implementation

Now it's your turn to implement the `generate_image_pipeline` function. Use the `pipe` and `controlnet` objects provided in the setup cell, along with the helper functions for preprocessing and post-processing. Remember to adhere to all the requirements listed above.

```python
def generate_image_pipeline(prompt: str, control_image: Image.Image, negative_prompt: str = "", seed: int = None, steps: int = 25) -> Image.Image:
    # Your code here
    pass
```

After implementing the function, test it with the provided `input_control_image` and a creative prompt.


In [ ]:
import torch
from PIL import Image
import numpy as np
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from controlnet_aux import CannyDetector
import os

# --- Configuration and Mock Setup (2026 Ready) ---

# Set device to CUDA if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Mock model paths (in a real scenario, these would be actual Hugging Face model IDs or local paths)
BASE_MODEL_ID = "stabilityai/stable-diffusion-3-medium" # A powerful 2026-era foundation model
CONTROLNET_MODEL_ID = "lllyasviel/controlnet-sd3-canny" # Hypothetical SD3-compatible ControlNet
LORA_MODEL_ID = "agenticlabs/sd3-style-vaporwave-lora" # Hypothetical custom LoRA

# --- Helper Functions for Mocking and Preprocessing ---

def mock_load_model(model_id, model_type):
    """Mocks loading a model. In a real scenario, this would use from_pretrained."""
    print(f"[MOCK] Loading {model_type} from {model_id}...")
    # Simulate a dummy object that can be passed around
    class MockModel:
        def __init__(self, name):
            self.name = name
            self.device = device
            self.dtype = torch.float16 if device == "cuda" else torch.float32
        def to(self, device, dtype=None):
            self.device = device
            if dtype: self.dtype = dtype
            return self
        def __call__(self, *args, **kwargs):
            print(f"[MOCK] {self.name} called.")
            return None # Or a dummy tensor if needed

    if model_type == "pipeline":
        # For the pipeline, we need a more complex mock that can simulate generation
        class MockPipeline(MockModel):
            def __init__(self, name, controlnet, lora):
                super().__init__(name)
                self.controlnet = controlnet
                self.lora = lora
                self.scheduler = UniPCMultistepScheduler()
                self.vae = MockModel("MockVAE")
                self.text_encoder = MockModel("MockTextEncoder")
                self.unet = MockModel("MockUNet")

            def load_lora_weights(self, lora_id):
                print(f"[MOCK] Loading LoRA weights {lora_id} into pipeline.")

            def __call__(self, prompt, image, negative_prompt, num_inference_steps, generator, **kwargs):
                print(f"[MOCK] Generating image with prompt: '{prompt}' and control image.")
                # Simulate image generation: create a black image
                dummy_image = Image.new('RGB', (768, 768), color = 'black')
                return [dummy_image]
        return MockPipeline(model_id, mock_load_model(CONTROLNET_MODEL_ID, "controlnet"), mock_load_model(LORA_MODEL_ID, "lora"))
    elif model_type == "controlnet":
        return ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16) if device == "cuda" else ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny")
    elif model_type == "lora":
        return MockModel(model_id) # LoRA is typically loaded into the pipeline directly
    else:
        return MockModel(model_id)


def preprocess_control_image(image: Image.Image) -> Image.Image:
    """Applies Canny edge detection to the control image."""
    print("[PREPROCESS] Applying Canny edge detection...")
    canny_detector = CannyDetector()
    processed_image = canny_detector(image)
    return processed_image

def mock_upscale_image(image: Image.Image) -> Image.Image:
    """Mocks an image upscaling process."""
    print("[POSTPROCESS] Mock Upscaling image...")
    # In a real scenario, this would use a dedicated upscaler model (e.g., ESRGAN, SwinIR)
    # For this exercise, we'll just resize it to double the dimensions.
    upscaled_image = image.resize((image.width * 2, image.height * 2), Image.LANCZOS)
    return upscaled_image

def mock_add_watermark(image: Image.Image, text: str = "AgenticLabs.ng") -> Image.Image:
    """Mocks adding a simple watermark to the image."""
    print("[POSTPROCESS] Adding watermark...")
    from PIL import ImageDraw, ImageFont
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    try:
        font = ImageFont.truetype("arial.ttf", 30)
    except IOError:
        font = ImageFont.load_default()
    text_width, text_height = draw.textsize(text, font)
    x = img_copy.width - text_width - 10
    y = img_copy.height - text_height - 10
    draw.text((x, y), text, font=font, fill=(255, 255, 255, 128)) # White, semi-transparent
    return img_copy

# --- Load Mock ControlNet and Pipeline (for student's use) ---

# Load ControlNet model
# In a real scenario:
# controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
# For this exercise, we use the actual Canny ControlNet for preprocessing, but the pipeline will be mocked.
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
controlnet.to(device)

# Load the base pipeline with ControlNet
# In a real scenario:
# pipe = StableDiffusionControlNetPipeline.from_pretrained(
#     BASE_MODEL_ID,
#     controlnet=controlnet,
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#     safety_checker=None # Disable for faster inference if not needed in production
# )
# For this exercise, we use a mock pipeline to simulate generation without downloading large models.
pipe = mock_load_model(BASE_MODEL_ID, "pipeline")
pipe.controlnet = controlnet # Assign the actual controlnet to the mock pipe
pipe.to(device)

# Load LoRA weights into the pipeline
# In a real scenario:
# pipe.load_lora_weights(LORA_MODEL_ID)
pipe.load_lora_weights(LORA_MODEL_ID) # Mock call

# Set scheduler for faster inference
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

print("Setup complete. `pipe` and `controlnet` are ready for use.")

# --- Example Control Image (for student's use) ---
# Download a sample image for ControlNet input
control_image_url = "https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd_controlnet/person.png"
input_control_image = load_image(control_image_url).resize((768, 768))
print(f"Loaded example control image of size: {input_control_image.size}")


# --- Reference Solution ---

def generate_image_pipeline(
    prompt: str,
    control_image: Image.Image,
    negative_prompt: str = "",
    seed: int = None,
    steps: int = 25
) -> Image.Image:
    """
    Generates an image using a Stable Diffusion pipeline with ControlNet and LoRA,
    followed by post-processing steps.

    Args:
        prompt (str): The positive text prompt for image generation.
        control_image (PIL.Image.Image): The input image for ControlNet (e.g., a pose image).
        negative_prompt (str, optional): The negative text prompt. Defaults to "".
        seed (int, optional): Random seed for reproducibility. Defaults to None.
        steps (int, optional): Number of inference steps. Defaults to 25.

    Returns:
        PIL.Image.Image: The final post-processed generated image.
    """
    print("\n--- Starting Image Generation Pipeline ---")

    # 1. Preprocess Control Image
    # Apply Canny edge detection to the input control image.
    processed_control_image = preprocess_control_image(control_image)
    print(f"Control image preprocessed. Output size: {processed_control_image.size}")

    # 2. Set up Random Generator for Reproducibility
    # If a seed is provided, create a torch generator for consistent results.
    generator = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
    print(f"Generator set with seed: {seed}" if seed is not None else "No specific seed provided.")

    # 3. Perform Image Generation with ControlNet and LoRA
    # The `pipe` object already has ControlNet integrated and LoRA weights loaded.
    # We pass the preprocessed control image to the pipeline.
    print(f"Generating image with {steps} inference steps...")
    output = pipe(
        prompt=prompt,
        image=processed_control_image, # This is the preprocessed control image
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        generator=generator,
        # Additional parameters can be added here, e.g., guidance_scale, controlnet_conditioning_scale
        # guidance_scale=7.5,
        # controlnet_conditioning_scale=1.0
    ).images[0]
    print("Base image generation complete.")

    # 4. Post-processing Steps
    # Apply a series of enhancements to the generated image.
    final_image = output

    # Example Post-processing 1: Upscaling
    final_image = mock_upscale_image(final_image)
    print(f"Image upscaled. New size: {final_image.size}")

    # Example Post-processing 2: Adding a watermark
    final_image = mock_add_watermark(final_image, text="AgenticLabs.ng | SD02-L08")
    print("Watermark added.")

    print("--- Image Generation Pipeline Complete ---")
    return final_image

# --- Test the Pipeline ---

# Define a creative prompt
my_prompt = "A futuristic cyberpunk city street at night, neon lights reflecting on wet pavement, a lone figure walking, highly detailed, cinematic lighting, vaporwave style"
my_negative_prompt = "blurry, low quality, bad anatomy, deformed, ugly, cartoon, sketch, monochrome"
my_seed = 42 # For reproducibility

print("\n--- Running the pipeline with example inputs ---")
generated_final_image = generate_image_pipeline(
    prompt=my_prompt,
    control_image=input_control_image,
    negative_prompt=my_negative_prompt,
    seed=my_seed,
    steps=30
)

print("\nPipeline execution finished. Displaying mock final image.")
# In a real notebook, you would display the image here:
# generated_final_image.save("final_generated_image.png")
# display(generated_final_image)

# For this mock, we'll just confirm its type and size
print(f"Generated image type: {type(generated_final_image)}")
print(f"Generated image size: {generated_final_image.size}")
